For every true pair:
Exact signals
- raw name exact
- normalized name exact
- raw address exact
- normalized address exact
- country exact
- normalized name + country
- normalized address + country
- normalized name + normalized address
- normalized name OR normalized address
Similarity signals
- name character similarity
- name token Jaccard
- address character similarity
- address token Jaccard
Important diagnostics
- S2 vs S3 separately
- pairs with exact name
- pairs with exact address
- pairs missed by both exact name and exact address
- missing-address true pairs
- similarity distributions
- true-pair feature dataset
- blocking-coverage summary

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(
    r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train"
)

GROUND_TRUTH_PATH = BASE_DIR / "train_ground_truth.tsv"
SOURCE1_PATH = BASE_DIR / "train_source1.tsv"
SOURCE2_PATH = BASE_DIR / "train_source2.tsv"
SOURCE3_PATH = BASE_DIR / "train_source3.tsv"

OUTPUT_DIR = Path(
    r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\student_resource\amazon_er\outputs\E001_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(value):
    """
    Conservative normalization.

    1. Unicode NFKC
    2. Casefold
    3. Punctuation -> space
    4. Collapse whitespace
    """

    if pd.isna(value):
        return ""

    text = str(value)

    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith("P"):
            chars.append(" ")
        else:
            chars.append(char)

    text = "".join(chars)

    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize(text):
    if not text:
        return set()

    return set(text.split())


# ============================================================
# JACCARD SIMILARITY
# ============================================================

def jaccard_similarity(a, b):

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    intersection = len(a & b)
    union = len(a | b)

    if union == 0:
        return 0.0

    return intersection / union


# ============================================================
# CHARACTER N-GRAMS
# ============================================================

def char_ngrams(text, n=3):

    if not text:
        return set()

    # Pad slightly so short strings can still generate useful grams
    if len(text) < n:
        return {text}

    return {
        text[i:i + n]
        for i in range(len(text) - n + 1)
    }


# ============================================================
# LEVENSHTEIN SIMILARITY
# ============================================================

def levenshtein_similarity(a, b):

    if a == b:
        return 1.0

    if not a or not b:
        return 0.0

    # Keep the shorter string on the columns
    if len(a) < len(b):
        a, b = b, a

    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):

        current = [i]

        for j, char_b in enumerate(b, start=1):

            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            substitute_cost = (
                previous[j - 1]
                + (char_a != char_b)
            )

            current.append(
                min(
                    insert_cost,
                    delete_cost,
                    substitute_cost,
                )
            )

        previous = current

    distance = previous[-1]

    max_length = max(len(a), len(b))

    if max_length == 0:
        return 1.0

    return 1.0 - distance / max_length


# ============================================================
# TEXT PREPARATION
# ============================================================

def prepare_source(df):

    df = df.copy()

    df["business_name"] = (
        df["business_name"]
        .fillna("")
        .astype(str)
    )

    df["business_address"] = (
        df["business_address"]
        .fillna("")
        .astype(str)
    )

    df["country"] = (
        df["country"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["business_name_normalized"] = (
        df["business_name"]
        .map(normalize_text)
    )

    df["business_address_normalized"] = (
        df["business_address"]
        .map(normalize_text)
    )

    return df


# ============================================================
# LOAD GROUND TRUTH
# ============================================================

def load_ground_truth():

    print("=" * 70)
    print("LOADING GROUND TRUTH")
    print("=" * 70)

    gt = pd.read_csv(
        GROUND_TRUTH_PATH,
        sep="\t",
        dtype=str,
        keep_default_na=False,
    )

    print(f"Rows: {len(gt):,}")

    return gt


# ============================================================
# EXPAND GROUND TRUTH INTO TRUE PAIRS
# ============================================================

def expand_ground_truth(gt):

    print()
    print("=" * 70)
    print("EXPANDING GROUND TRUTH INTO TRUE PAIRS")
    print("=" * 70)

    records = []

    for _, row in gt.iterrows():

        s1_id = row["source1_entity_id"]

        matched_ids = str(
            row["matched_entity_ids"]
        ).strip()

        if not matched_ids:
            continue

        for matched_id in matched_ids.split(","):

            matched_id = matched_id.strip()

            if not matched_id:
                continue

            if matched_id.startswith("S2-"):
                source = "S2"

            elif matched_id.startswith("S3-"):
                source = "S3"

            else:
                source = "INVALID"

            records.append(
                {
                    "source1_entity_id": s1_id,
                    "matched_entity_id": matched_id,
                    "matched_source": source,
                }
            )

    pairs = pd.DataFrame(records)

    print(f"True pairs: {len(pairs):,}")

    print()
    print(
        pairs["matched_source"]
        .value_counts()
        .to_string()
    )

    return pairs


# ============================================================
# LOAD SOURCE DATA
# ============================================================

def load_source(path, source_name):

    print()
    print("=" * 70)
    print(f"LOADING {source_name}")
    print("=" * 70)

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
    )

    print(f"Rows: {len(df):,}")

    df = prepare_source(df)

    return df


# ============================================================
# CREATE LOOKUP TABLES
# ============================================================

def create_lookup(df):

    columns = [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized",
    ]

    lookup = (
        df[columns]
        .drop_duplicates("entity_id")
        .set_index("entity_id")
    )

    return lookup


# ============================================================
# CALCULATE PAIR FEATURES
# ============================================================

def calculate_pair_features(
    pairs,
    s1_lookup,
    s2_lookup,
    s3_lookup,
):

    print()
    print("=" * 70)
    print("CALCULATING TRUE-PAIR FEATURES")
    print("=" * 70)

    results = []

    missing_s1 = 0
    missing_target = 0

    for idx, pair in enumerate(
        pairs.itertuples(index=False),
        start=1
    ):

        s1_id = pair.source1_entity_id
        target_id = pair.matched_entity_id
        target_source = pair.matched_source

        # ----------------------------------------------------
        # Retrieve S1
        # ----------------------------------------------------

        if s1_id not in s1_lookup.index:
            missing_s1 += 1
            continue

        s1 = s1_lookup.loc[s1_id]

        # ----------------------------------------------------
        # Retrieve target
        # ----------------------------------------------------

        if target_source == "S2":
            target_lookup = s2_lookup

        elif target_source == "S3":
            target_lookup = s3_lookup

        else:
            continue

        if target_id not in target_lookup.index:
            missing_target += 1
            continue

        target = target_lookup.loc[target_id]

        # ----------------------------------------------------
        # Values
        # ----------------------------------------------------

        s1_name = s1["business_name"]
        target_name = target["business_name"]

        s1_address = s1["business_address"]
        target_address = target["business_address"]

        s1_name_norm = s1["business_name_normalized"]
        target_name_norm = target[
            "business_name_normalized"
        ]

        s1_address_norm = (
            s1["business_address_normalized"]
        )

        target_address_norm = (
            target["business_address_normalized"]
        )

        s1_country = s1["country"]
        target_country = target["country"]

        # ----------------------------------------------------
        # Exact comparisons
        # ----------------------------------------------------

        raw_name_exact = (
            bool(s1_name)
            and bool(target_name)
            and s1_name == target_name
        )

        normalized_name_exact = (
            bool(s1_name_norm)
            and bool(target_name_norm)
            and s1_name_norm == target_name_norm
        )

        raw_address_exact = (
            bool(s1_address)
            and bool(target_address)
            and s1_address == target_address
        )

        normalized_address_exact = (
            bool(s1_address_norm)
            and bool(target_address_norm)
            and s1_address_norm == target_address_norm
        )

        country_exact = (
            bool(s1_country)
            and bool(target_country)
            and s1_country.casefold()
            == target_country.casefold()
        )

        # ----------------------------------------------------
        # Combined exact signals
        # ----------------------------------------------------

        name_country_exact = (
            normalized_name_exact
            and country_exact
        )

        address_country_exact = (
            normalized_address_exact
            and country_exact
        )

        name_address_exact = (
            normalized_name_exact
            and normalized_address_exact
        )

        name_or_address_exact = (
            normalized_name_exact
            or normalized_address_exact
        )

        # ----------------------------------------------------
        # Token sets
        # ----------------------------------------------------

        name_tokens_a = tokenize(s1_name_norm)
        name_tokens_b = tokenize(target_name_norm)

        address_tokens_a = tokenize(
            s1_address_norm
        )

        address_tokens_b = tokenize(
            target_address_norm
        )

        # ----------------------------------------------------
        # Token Jaccard
        # ----------------------------------------------------

        name_token_jaccard = jaccard_similarity(
            name_tokens_a,
            name_tokens_b,
        )

        address_token_jaccard = jaccard_similarity(
            address_tokens_a,
            address_tokens_b,
        )

        # ----------------------------------------------------
        # Character n-gram Jaccard
        # ----------------------------------------------------

        name_ngrams_a = char_ngrams(
            s1_name_norm,
            n=3,
        )

        name_ngrams_b = char_ngrams(
            target_name_norm,
            n=3,
        )

        address_ngrams_a = char_ngrams(
            s1_address_norm,
            n=3,
        )

        address_ngrams_b = char_ngrams(
            target_address_norm,
            n=3,
        )

        name_char_jaccard = jaccard_similarity(
            name_ngrams_a,
            name_ngrams_b,
        )

        address_char_jaccard = jaccard_similarity(
            address_ngrams_a,
            address_ngrams_b,
        )

        # ----------------------------------------------------
        # Levenshtein
        # ----------------------------------------------------

        name_levenshtein = levenshtein_similarity(
            s1_name_norm,
            target_name_norm,
        )

        address_levenshtein = levenshtein_similarity(
            s1_address_norm,
            target_address_norm,
        )

        # ----------------------------------------------------
        # Missingness
        # ----------------------------------------------------

        s1_address_missing = not bool(
            s1_address.strip()
        )

        target_address_missing = not bool(
            target_address.strip()
        )

        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        results.append(
            {
                "source1_entity_id": s1_id,
                "matched_entity_id": target_id,
                "matched_source": target_source,

                "s1_business_name": s1_name,
                "matched_business_name": target_name,

                "s1_business_address": s1_address,
                "matched_business_address": target_address,

                "s1_country": s1_country,
                "matched_country": target_country,

                "raw_name_exact":
                    raw_name_exact,

                "normalized_name_exact":
                    normalized_name_exact,

                "raw_address_exact":
                    raw_address_exact,

                "normalized_address_exact":
                    normalized_address_exact,

                "country_exact":
                    country_exact,

                "name_country_exact":
                    name_country_exact,

                "address_country_exact":
                    address_country_exact,

                "name_address_exact":
                    name_address_exact,

                "name_or_address_exact":
                    name_or_address_exact,

                "name_token_jaccard":
                    name_token_jaccard,

                "address_token_jaccard":
                    address_token_jaccard,

                "name_char_jaccard":
                    name_char_jaccard,

                "address_char_jaccard":
                    address_char_jaccard,

                "name_levenshtein":
                    name_levenshtein,

                "address_levenshtein":
                    address_levenshtein,

                "s1_address_missing":
                    s1_address_missing,

                "matched_address_missing":
                    target_address_missing,
            }
        )

        if idx % 500_000 == 0:
            print(
                f"Processed {idx:,} / "
                f"{len(pairs):,} pairs"
            )

    result_df = pd.DataFrame(results)

    print()
    print(f"Generated feature rows : {len(result_df):,}")
    print(f"Missing S1 records     : {missing_s1:,}")
    print(
        f"Missing target records: "
        f"{missing_target:,}"
    )

    return result_df


# ============================================================
# COVERAGE ANALYSIS
# ============================================================

def calculate_coverage(df):

    print()
    print("=" * 70)
    print("BLOCKING SIGNAL COVERAGE")
    print("=" * 70)

    rules = {

        "raw_name_exact":
            "raw_name_exact",

        "normalized_name_exact":
            "normalized_name_exact",

        "raw_address_exact":
            "raw_address_exact",

        "normalized_address_exact":
            "normalized_address_exact",

        "country_exact":
            "country_exact",

        "name_country_exact":
            "name_country_exact",

        "address_country_exact":
            "address_country_exact",

        "name_address_exact":
            "name_address_exact",

        "name_or_address_exact":
            "name_or_address_exact",
    }

    rows = []

    total = len(df)

    for rule_name, column in rules.items():

        count = int(df[column].sum())

        rows.append(
            {
                "rule": rule_name,
                "true_pairs_retrieved": count,
                "total_true_pairs": total,
                "pair_recall_percent":
                    count / total * 100,
            }
        )

    coverage = pd.DataFrame(rows)

    print(
        coverage.to_string(index=False)
    )

    coverage.to_csv(
        OUTPUT_DIR / "blocking_signal_coverage.csv",
        index=False,
    )

    return coverage


# ============================================================
# SOURCE-SPECIFIC COVERAGE
# ============================================================

def calculate_source_coverage(df):

    print()
    print("=" * 70)
    print("SOURCE-SPECIFIC COVERAGE")
    print("=" * 70)

    rules = [
        "raw_name_exact",
        "normalized_name_exact",
        "raw_address_exact",
        "normalized_address_exact",
        "country_exact",
        "name_country_exact",
        "address_country_exact",
        "name_address_exact",
        "name_or_address_exact",
    ]

    rows = []

    for source in ["S2", "S3"]:

        subset = df[
            df["matched_source"] == source
        ]

        total = len(subset)

        for rule in rules:

            count = int(
                subset[rule].sum()
            )

            rows.append(
                {
                    "source": source,
                    "rule": rule,
                    "true_pairs": total,
                    "retrieved": count,
                    "recall_percent":
                        count / total * 100,
                }
            )

    result = pd.DataFrame(rows)

    print(
        result.to_string(index=False)
    )

    result.to_csv(
        OUTPUT_DIR / "source_specific_coverage.csv",
        index=False,
    )

    return result


# ============================================================
# RULE OVERLAP
# ============================================================

def calculate_rule_overlap(df):

    print()
    print("=" * 70)
    print("EXACT NAME / ADDRESS RULE OVERLAP")
    print("=" * 70)

    name = df["normalized_name_exact"]
    address = df["normalized_address_exact"]

    overlap = pd.DataFrame(
        {
            "category": [
                "Name AND Address",
                "Name only",
                "Address only",
                "Neither",
            ],
            "true_pair_count": [
                int((name & address).sum()),
                int((name & ~address).sum()),
                int((~name & address).sum()),
                int((~name & ~address).sum()),
            ],
        }
    )

    overlap["percentage"] = (
        overlap["true_pair_count"]
        / len(df)
        * 100
    )

    print(
        overlap.to_string(index=False)
    )

    overlap.to_csv(
        OUTPUT_DIR / "name_address_overlap.csv",
        index=False,
    )

    return overlap


# ============================================================
# DIFFICULT TRUE PAIRS
# ============================================================

def extract_difficult_pairs(df):

    print()
    print("=" * 70)
    print("DIFFICULT TRUE PAIRS")
    print("=" * 70)

    difficult = df[
        (
            ~df["normalized_name_exact"]
        )
        &
        (
            ~df["normalized_address_exact"]
        )
    ].copy()

    print(
        "True pairs missed by both "
        "exact normalized name and "
        "exact normalized address:"
    )

    print(
        f"{len(difficult):,} "
        f"({len(difficult) / len(df) * 100:.2f}%)"
    )

    # Save a manageable sample
    difficult_sample = difficult.sample(
        n=min(1000, len(difficult)),
        random_state=42,
    )

    difficult_sample.to_csv(
        OUTPUT_DIR / "difficult_true_pairs_sample.csv",
        index=False,
    )

    return difficult


# ============================================================
# MISSING ADDRESS ANALYSIS
# ============================================================

def analyze_missing_addresses(df):

    print()
    print("=" * 70)
    print("MISSING ADDRESS TRUE-PAIR ANALYSIS")
    print("=" * 70)

    both_missing = (
        df["s1_address_missing"]
        &
        df["matched_address_missing"]
    )

    either_missing = (
        df["s1_address_missing"]
        |
        df["matched_address_missing"]
    )

    neither_missing = ~either_missing

    print(
        f"Both addresses missing : "
        f"{both_missing.sum():,}"
    )

    print(
        f"Either address missing : "
        f"{either_missing.sum():,}"
    )

    print(
        f"Neither missing        : "
        f"{neither_missing.sum():,}"
    )

    result = pd.DataFrame(
        {
            "category": [
                "Both missing",
                "Either missing",
                "Neither missing",
            ],
            "true_pair_count": [
                int(both_missing.sum()),
                int(either_missing.sum()),
                int(neither_missing.sum()),
            ],
        }
    )

    result["percentage"] = (
        result["true_pair_count"]
        / len(df)
        * 100
    )

    result.to_csv(
        OUTPUT_DIR / "missing_address_true_pairs.csv",
        index=False,
    )


# ============================================================
# SIMILARITY STATISTICS
# ============================================================

def similarity_statistics(df):

    print()
    print("=" * 70)
    print("SIMILARITY DISTRIBUTIONS")
    print("=" * 70)

    similarity_columns = [
        "name_token_jaccard",
        "address_token_jaccard",
        "name_char_jaccard",
        "address_char_jaccard",
        "name_levenshtein",
        "address_levenshtein",
    ]

    rows = []

    for column in similarity_columns:

        series = df[column].dropna()

        rows.append(
            {
                "feature": column,
                "min":
                    series.min(),
                "p01":
                    series.quantile(0.01),
                "p05":
                    series.quantile(0.05),
                "p10":
                    series.quantile(0.10),
                "p25":
                    series.quantile(0.25),
                "median":
                    series.median(),
                "p75":
                    series.quantile(0.75),
                "p90":
                    series.quantile(0.90),
                "p95":
                    series.quantile(0.95),
                "p99":
                    series.quantile(0.99),
                "max":
                    series.max(),
                "mean":
                    series.mean(),
            }
        )

    stats = pd.DataFrame(rows)

    print(
        stats.to_string(index=False)
    )

    stats.to_csv(
        OUTPUT_DIR / "similarity_statistics.csv",
        index=False,
    )

    return stats


# ============================================================
# LOW-SIMILARITY TRUE PAIRS
# ============================================================

def analyze_low_similarity(df):

    print()
    print("=" * 70)
    print("LOW-SIMILARITY TRUE PAIRS")
    print("=" * 70)

    # True pairs where both exact name and exact
    # address failed.

    difficult = df[
        ~df["normalized_name_exact"]
        &
        ~df["normalized_address_exact"]
    ].copy()

    if difficult.empty:
        print("No difficult pairs found.")
        return

    # Lowest name similarity
    lowest_name = (
        difficult
        .sort_values("name_char_jaccard")
        .head(100)
    )

    lowest_name.to_csv(
        OUTPUT_DIR / "lowest_name_similarity_pairs.csv",
        index=False,
    )

    # Lowest address similarity
    lowest_address = (
        difficult
        .sort_values("address_char_jaccard")
        .head(100)
    )

    lowest_address.to_csv(
        OUTPUT_DIR / "lowest_address_similarity_pairs.csv",
        index=False,
    )

    print()
    print("Lowest name similarity examples:")

    print(
        lowest_name[
            [
                "source1_entity_id",
                "matched_entity_id",
                "matched_source",
                "s1_business_name",
                "matched_business_name",
                "name_char_jaccard",
                "name_levenshtein",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


# ============================================================
# MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # 1. Load ground truth
    # --------------------------------------------------------

    gt = load_ground_truth()

    # --------------------------------------------------------
    # 2. Expand into true pairs
    # --------------------------------------------------------

    pairs = expand_ground_truth(gt)

    # --------------------------------------------------------
    # 3. Load sources
    # --------------------------------------------------------

    s1 = load_source(
        SOURCE1_PATH,
        "SOURCE 1",
    )

    s2 = load_source(
        SOURCE2_PATH,
        "SOURCE 2",
    )

    s3 = load_source(
        SOURCE3_PATH,
        "SOURCE 3",
    )

    # --------------------------------------------------------
    # 4. Create lookups
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("CREATING ENTITY LOOKUPS")
    print("=" * 70)

    s1_lookup = create_lookup(s1)
    s2_lookup = create_lookup(s2)
    s3_lookup = create_lookup(s3)

    print(f"S1 lookup: {len(s1_lookup):,}")
    print(f"S2 lookup: {len(s2_lookup):,}")
    print(f"S3 lookup: {len(s3_lookup):,}")

    # --------------------------------------------------------
    # 5. Calculate pairwise features
    # --------------------------------------------------------

    pair_features = calculate_pair_features(
        pairs,
        s1_lookup,
        s2_lookup,
        s3_lookup,
    )

    # --------------------------------------------------------
    # 6. Save complete true-pair dataset
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("SAVING TRUE-PAIR FEATURE DATASET")
    print("=" * 70)

    pair_features.to_csv(
        OUTPUT_DIR / "true_pair_features.csv",
        index=False,
    )

    print(
        f"Saved: "
        f"{OUTPUT_DIR / 'true_pair_features.csv'}"
    )

    # --------------------------------------------------------
    # 7. Coverage
    # --------------------------------------------------------

    calculate_coverage(
        pair_features
    )

    calculate_source_coverage(
        pair_features
    )

    calculate_rule_overlap(
        pair_features
    )

    # --------------------------------------------------------
    # 8. Difficult cases
    # --------------------------------------------------------

    difficult = extract_difficult_pairs(
        pair_features
    )

    analyze_missing_addresses(
        pair_features
    )

    # --------------------------------------------------------
    # 9. Similarity
    # --------------------------------------------------------

    similarity_statistics(
        pair_features
    )

    analyze_low_similarity(
        pair_features
    )

    # --------------------------------------------------------
    # 10. Final summary
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("TRUE-PAIR ANALYSIS COMPLETE")
    print("=" * 70)

    print(
        f"True pairs analyzed : "
        f"{len(pair_features):,}"
    )

    print(
        f"S2 pairs           : "
        f"{(
            pair_features['matched_source'] == 'S2'
        ).sum():,}"
    )

    print(
        f"S3 pairs           : "
        f"{(
            pair_features['matched_source'] == 'S3'
        ).sum():,}"
    )

    print(
        f"Difficult pairs     : "
        f"{len(difficult):,}"
    )

    print()
    print("Output files:")

    for file in sorted(
        OUTPUT_DIR.glob("*.csv")
    ):
        print(f"  {file.name}")


if __name__ == "__main__":
    main()

LOADING GROUND TRUTH
Rows: 2,206,821

EXPANDING GROUND TRUTH INTO TRUE PAIRS
True pairs: 7,638,365

matched_source
S3    3944746
S2    3693619

LOADING SOURCE 1
Rows: 2,206,821

LOADING SOURCE 2
Rows: 5,034,616

LOADING SOURCE 3
Rows: 5,285,603

CREATING ENTITY LOOKUPS
S1 lookup: 2,206,821
S2 lookup: 5,034,616
S3 lookup: 5,285,603

CALCULATING TRUE-PAIR FEATURES
Processed 500,000 / 7,638,365 pairs
Processed 1,000,000 / 7,638,365 pairs
Processed 1,500,000 / 7,638,365 pairs
Processed 2,000,000 / 7,638,365 pairs
Processed 2,500,000 / 7,638,365 pairs
Processed 3,000,000 / 7,638,365 pairs
Processed 3,500,000 / 7,638,365 pairs
Processed 4,000,000 / 7,638,365 pairs
Processed 4,500,000 / 7,638,365 pairs
Processed 5,000,000 / 7,638,365 pairs
Processed 5,500,000 / 7,638,365 pairs
Processed 6,000,000 / 7,638,365 pairs
Processed 6,500,000 / 7,638,365 pairs
Processed 7,000,000 / 7,638,365 pairs
Processed 7,500,000 / 7,638,365 pairs

Generated feature rows : 7,638,365
Missing S1 records     : 0
Miss